In [1]:
# === セル1: パッケージインストール ===
!pip install -q requests beautifulsoup4 lxml pandas numpy sentence-transformers rank-bm25 transformers accelerate torch ipywidgets ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.3 MB/s eta 0:00:00


In [2]:
from urllib.parse import quote

import re, json, time, math, textwrap, requests, os
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [3]:
USER_AGENT = "BaseballStatsAI/4.1 (Jupyter; research)"
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8"
}
REQUEST_TIMEOUT = 20
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
WIKIPEDIA_API_JA = "https://ja.wikipedia.org/w/api.php"
WIKIPEDIA_API_EN = "https://en.wikipedia.org/w/api.php"
MLB_STATS_BASE = "https://statsapi.mlb.com/api/v1"

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1"

DEFAULT_SEASON = datetime.now().year

In [4]:
# ─────────────────────────────────────
# 1. ユーティリティ
# ─────────────────────────────────────
def safe_get(url, params=None, headers=None, timeout=REQUEST_TIMEOUT):
    h = HEADERS.copy()
    if headers:
        h.update(headers)
    r = requests.get(url, params=params, headers=h, timeout=timeout)
    r.raise_for_status()
    return r

def normalize_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_name_key(text):
    text = normalize_text(text).lower()
    text = text.replace("・", "").replace("·", "").replace("'", "").replace("’", "")
    text = re.sub(r"[\s\-_]+", "", text)
    return text

def tokenize_jp_en(text):
    text = normalize_text(text).lower()
    text = re.sub(r"[^\wぁ-んァ-ヴー一-龠々]+", " ", text)
    return [t for t in text.split() if t]

def truncate_text(text, max_len=1200):
    text = normalize_text(text)
    return text if len(text) <= max_len else text[:max_len] + "..."

def to_markdown_table(df):
    if df is None or df.empty:
        return "データなし"
    return df.to_markdown(index=False)

def make_section(title, body):
    return f"## {title}\n\n{body}\n"

def safe_num(x):
    if x in (None, "", ".---", "-.--", "--", "—"):
        return None
    try:
        s = normalize_text(x).replace(",", "")
        if s in {"-.---", ".---"}:
            return None
        return float(s)
    except Exception:
        return None

def extract_year_candidates(text):
    text = normalize_text(text)
    years = re.findall(r"(19\d{2}|20\d{2})", text)
    return [int(y) for y in years]

def row_matches_season(cells, season):
    if season in (None, "", 0):
        return True
    season = int(season)
    joined = " ".join([normalize_text(c) for c in cells if c is not None])
    years = extract_year_candidates(joined)
    return season in years

def generate_name_candidates(player_name, wiki_page=None):
    candidates = []
    base = normalize_text(player_name)
    if base:
        candidates.append(base)
    if wiki_page:
        title = normalize_text(wiki_page.get("title", ""))
        if title:
            candidates.append(title)
        extract = normalize_text(wiki_page.get("extract", ""))[:300]
        paren_variants = re.findall(r"([A-Za-z][A-Za-z .'-]{2,})", title + " " + extract)
        for item in paren_variants:
            item = normalize_text(item)
            if item:
                candidates.append(item)
    uniq = []
    seen = set()
    for item in candidates:
        key = normalize_name_key(item)
        if key and key not in seen:
            seen.add(key)
            uniq.append(item)
    return uniq

def score_name_match(query_name, candidate_text):
    q = normalize_name_key(query_name)
    c = normalize_name_key(candidate_text)
    if not q or not c:
        return 0
    if q == c:
        return 100
    if q in c or c in q:
        return 70
    q_tokens = set(tokenize_jp_en(query_name))
    c_tokens = set(tokenize_jp_en(candidate_text))
    overlap = len(q_tokens & c_tokens)
    if overlap:
        return 25 + overlap * 10
    return 0

print("✅ ユーティリティ定義完了")

✅ ユーティリティ定義完了


In [5]:
# ─────────────────────────────────────
# 2. Wikipedia 取得
# ─────────────────────────────────────
def wikipedia_search(query, lang="ja", limit=5):
    api = WIKIPEDIA_API_JA if lang == "ja" else WIKIPEDIA_API_EN
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "utf8": 1,
        "format": "json",
        "srlimit": limit
    }
    try:
        return safe_get(api, params=params).json().get("query", {}).get("search", [])
    except Exception:
        return []

def wikipedia_page_extract(title, lang="ja"):
    api = WIKIPEDIA_API_JA if lang == "ja" else WIKIPEDIA_API_EN
    params = {
        "action": "query",
        "prop": "extracts|info",
        "titles": title,
        "explaintext": 1,
        "inprop": "url",
        "format": "json"
    }
    try:
        data = safe_get(api, params=params).json()
        pages = data.get("query", {}).get("pages", {})
        if not pages:
            return None
        page = next(iter(pages.values()))
        if "missing" in page:
            return None
        return {
            "title": page.get("title", ""),
            "extract": normalize_text(page.get("extract", "")),
            "fullurl": page.get("fullurl", ""),
            "lang": lang
        }
    except Exception:
        return None

def wikipedia_best_match(player_name):
    """日本語版を優先しつつ英語版も使って野球選手ページを高精度で選ぶ"""
    search_queries = [player_name]
    if not re.search(r"(野球|投手|内野手|外野手|捕手|選手)", player_name):
        search_queries.append(f"{player_name} 野球")
        search_queries.append(f"{player_name} baseball")

    candidates = []
    for lang in ("ja", "en"):
        for query in search_queries:
            results = wikipedia_search(query, lang=lang, limit=8)
            for r in results:
                title = normalize_text(r.get("title", ""))
                snippet = BeautifulSoup(r.get("snippet", ""), "html.parser").get_text(" ", strip=True)
                score = score_name_match(player_name, title)

                text_for_bonus = f"{title} {snippet}".lower()
                baseball_keywords = ["野球", "投手", "捕手", "内野手", "外野手", "プロ野球", "メジャーリーグ", "baseball", "pitcher", "outfielder", "infielder", "catcher"]
                for kw in baseball_keywords:
                    if kw.lower() in text_for_bonus:
                        score += 12

                if "曖昧さ回避" in text_for_bonus or "disambiguation" in text_for_bonus:
                    score -= 80

                candidates.append({
                    "lang": lang,
                    "title": title,
                    "score": score,
                    "snippet": snippet
                })

    if not candidates:
        direct = wikipedia_page_extract(player_name, lang="ja") or wikipedia_page_extract(player_name, lang="en")
        return direct

    candidates.sort(key=lambda x: x["score"], reverse=True)

    for cand in candidates[:8]:
        page = wikipedia_page_extract(cand["title"], lang=cand["lang"])
        if not page:
            continue
        validation_text = f"{page.get('title', '')} {page.get('extract', '')[:500]}"
        if score_name_match(player_name, validation_text) >= 35:
            page["lang"] = cand["lang"]
            return page

    best = candidates[0]
    page = wikipedia_page_extract(best["title"], lang=best["lang"])
    if page:
        page["lang"] = best["lang"]
    return page

print("✅ Wikipedia関数定義完了")

✅ Wikipedia関数定義完了


In [6]:
# ─────────────────────────────────────
# 3. MLB Stats API 取得
# ─────────────────────────────────────
def mlb_search_people(player_name):
    try:
        url = f"{MLB_STATS_BASE}/people/search"
        return safe_get(url, params={"names": player_name}).json().get("people", [])
    except Exception:
        return []

def mlb_get_person_detail(person_id, season=None):
    try:
        url = f"{MLB_STATS_BASE}/people/{person_id}"
        hydrate = "currentTeam,stats(group=[hitting,pitching,fielding],type=[season,career])"
        if season not in (None, "", 0):
            hydrate = f"currentTeam,stats(group=[hitting,pitching,fielding],type=[season,career],season={int(season)})"
        params = {"hydrate": hydrate}
        people = safe_get(url, params=params).json().get("people", [])
        return people[0] if people else None
    except Exception:
        return None

def extract_stats_blocks(detail):
    """stats ブロックを group×type のキーで辞書化"""
    result = {}
    for block in detail.get("stats", []):
        grp = block.get("group", {}).get("displayName", "").lower()
        typ = block.get("type", {}).get("displayName", "").lower()
        splits = block.get("splits", [])
        if splits:
            result[f"{grp}_{typ}"] = splits[0].get("stat", {})
    return result

def fetch_mlb_profile(player_name, season=None):
    people = mlb_search_people(player_name)
    if not people:
        return None

    def score_person(p):
        s = 0
        full = normalize_text(p.get("fullName", ""))
        s += score_name_match(player_name, full)
        if p.get("active"):
            s += 20
        primary_pos = normalize_text((p.get("primaryPosition") or {}).get("name", ""))
        if primary_pos:
            s += 3
        return s

    people.sort(key=score_person, reverse=True)
    best_person = people[0]
    detail = mlb_get_person_detail(best_person.get("id"), season=season) or best_person
    stats = extract_stats_blocks(detail)

    return {
        "source": "MLB",
        "id": detail.get("id"),
        "name": detail.get("fullName"),
        "birth_date": detail.get("birthDate"),
        "current_age": detail.get("currentAge"),
        "height": detail.get("height"),
        "weight": detail.get("weight"),
        "bat_side": (detail.get("batSide") or {}).get("description"),
        "pitch_hand": (detail.get("pitchHand") or {}).get("description"),
        "number": detail.get("primaryNumber"),
        "position": (detail.get("primaryPosition") or {}).get("name"),
        "active": detail.get("active"),
        "team": (detail.get("currentTeam") or {}).get("name"),
        "debut": detail.get("mlbDebutDate"),
        "nickname": detail.get("nickName"),
        "country": detail.get("birthCountry"),
        "city": detail.get("birthCity"),
        "season": int(season) if season not in (None, "", 0) else None,
        "season_hitting": stats.get("hitting_season", {}),
        "career_hitting": stats.get("hitting_career", {}),
        "season_pitching": stats.get("pitching_season", {}),
        "career_pitching": stats.get("pitching_career", {}),
        "season_fielding": stats.get("fielding_season", {}),
        "career_fielding": stats.get("fielding_career", {}),
        "raw": detail
    }

print("✅ MLB API関数定義完了")

✅ MLB API関数定義完了


In [7]:
# ─────────────────────────────────────
# 5. 選手バンドル構築（Ver.07 MLB専用）
# ─────────────────────────────────────
def build_player_bundle(player_name, season=None):
    """Wikipedia + MLB の情報をまとめて取得"""
    wiki = wikipedia_best_match(player_name)
    mlb = fetch_mlb_profile(player_name, season=season)

    docs = []

    if wiki:
        docs.append({
            "source": "Wikipedia",
            "title": wiki.get("title", ""),
            "url": wiki.get("fullurl", ""),
            "content": wiki.get("extract", "")
        })

    if mlb:
        parts = [
            f"氏名: {mlb.get('name', '')}",
            f"チーム: {mlb.get('team', '')}",
            f"ポジション: {mlb.get('position', '')}",
            f"年齢: {mlb.get('current_age', '')}",
            f"身長: {mlb.get('height', '')}",
            f"体重: {mlb.get('weight', '')}",
            f"投打: {mlb.get('pitch_hand', '')} / {mlb.get('bat_side', '')}",
            f"MLBデビュー: {mlb.get('debut', '')}",
            f"比較年度: {mlb.get('season', '')}",
            f"今季打撃: {json.dumps(mlb.get('season_hitting', {}), ensure_ascii=False)}",
            f"通算打撃: {json.dumps(mlb.get('career_hitting', {}), ensure_ascii=False)}",
            f"今季投球: {json.dumps(mlb.get('season_pitching', {}), ensure_ascii=False)}",
            f"通算投球: {json.dumps(mlb.get('career_pitching', {}), ensure_ascii=False)}",
            f"今季守備: {json.dumps(mlb.get('season_fielding', {}), ensure_ascii=False)}",
            f"通算守備: {json.dumps(mlb.get('career_fielding', {}), ensure_ascii=False)}",
        ]
        docs.append({
            "source": "MLB",
            "title": mlb.get("name", ""),
            "url": f"https://www.mlb.com/player/{mlb.get('id')}",
            "content": "\n".join(parts)
        })

    return {
        "player_name": player_name,
        "league": "MLB",
        "season": int(season) if season not in (None, "", 0) else None,
        "wikipedia": wiki,
        "mlb": mlb,
        "documents": docs
    }

print("✅ バンドル構築関数定義完了")

✅ バンドル構築関数定義完了


In [8]:
# ─────────────────────────────────────
# 6. 全野球指標の比較表生成
# ─────────────────────────────────────

MLB_HIT_FIELDS = [
    ("試合",         "gamesPlayed"),
    ("打席",         "plateAppearances"),
    ("打数",         "atBats"),
    ("得点",         "runs"),
    ("安打",         "hits"),
    ("二塁打",       "doubles"),
    ("三塁打",       "triples"),
    ("本塁打",       "homeRuns"),
    ("打点",         "rbi"),
    ("四球",         "baseOnBalls"),
    ("故意四球",     "intentionalWalks"),
    ("死球",         "hitByPitch"),
    ("三振",         "strikeOuts"),
    ("盗塁",         "stolenBases"),
    ("盗塁死",       "caughtStealing"),
    ("盗塁成功率",   "stolenBasePercentage"),
    ("犠打",         "sacBunts"),
    ("犠飛",         "sacFlies"),
    ("併殺打",       "groundIntoDoublePlay"),
    ("総塁打",       "totalBases"),
    ("残塁",         "leftOnBase"),
    ("打率 (AVG)",   "avg"),
    ("出塁率 (OBP)", "obp"),
    ("長打率 (SLG)", "slg"),
    ("OPS",          "ops"),
    ("BABIP",        "babip"),
    ("本塁打毎打数", "atBatsPerHomeRun"),
    ("GO/AO",        "groundOutsToAirouts"),
    ("投球数",       "numberOfPitches"),
]

MLB_PITCH_FIELDS = [
    ("試合",         "gamesPlayed"),
    ("先発",         "gamesStarted"),
    ("勝利",         "wins"),
    ("敗戦",         "losses"),
    ("セーブ",       "saves"),
    ("ホールド",     "holds"),
    ("投球回",       "inningsPitched"),
    ("被安打",       "hits"),
    ("失点",         "runs"),
    ("自責点",       "earnedRuns"),
    ("被本塁打",     "homeRuns"),
    ("四球",         "baseOnBalls"),
    ("死球",         "hitByPitch"),
    ("奪三振",       "strikeOuts"),
    ("暴投",         "wildPitches"),
    ("ボーク",       "balks"),
    ("防御率 (ERA)", "era"),
    ("WHIP",         "whip"),
    ("K/9",          "strikeoutsPer9Inn"),
    ("BB/9",         "walksPer9Inn"),
    ("H/9",          "hitsPer9Inn"),
    ("HR/9",         "homeRunsPer9"),
    ("K/BB",         "strikeoutWalkRatio"),
    ("被打率",       "avg"),
    ("完投",         "completeGames"),
    ("完封",         "shutouts"),
    ("QS",           "qualityStarts"),
    ("投球数",       "numberOfPitches"),
    ("ストライク%",  "strikePercentage"),
]

MLB_FIELD_FIELDS = [
    ("試合",            "gamesPlayed"),
    ("先発",            "gamesStarted"),
    ("刺殺",            "putOuts"),
    ("補殺",            "assists"),
    ("失策",            "errors"),
    ("守備機会",        "chances"),
    ("守備率 (FPCT)",   "fielding"),
    ("DP",              "doublePlays"),
    ("RF/G",            "rangeFactorPerGame"),
    ("RF/9",            "rangeFactorPer9Inn"),
    ("守備回",          "innings"),
]

def build_comparison_table(bundles, stat_key, fields_def):
    names = [b["player_name"] for b in bundles]
    rows = []
    for jp_label, key in fields_def:
        row = {"指標": jp_label}
        has_data = False
        for b in bundles:
            mlb = b.get("mlb") or {}
            stat = mlb.get(stat_key, {})
            val = stat.get(key, "") if isinstance(stat, dict) else ""
            if val not in (None, "", ".---", "-.--"):
                has_data = True
            row[b["player_name"]] = val if val is not None else ""
        if has_data:
            rows.append(row)
    return pd.DataFrame(rows, columns=["指標"] + names)

def build_basic_comparison(bundles):
    names = [b["player_name"] for b in bundles]
    rows = []
    fields = [
        ("名前",        lambda m: m.get("name")),
        ("チーム",      lambda m: m.get("team")),
        ("ポジション",  lambda m: m.get("position")),
        ("背番号",      lambda m: m.get("number")),
        ("年齢",        lambda m: m.get("current_age")),
        ("生年月日",    lambda m: m.get("birth_date")),
        ("出身国",      lambda m: m.get("country")),
        ("身長",        lambda m: m.get("height")),
        ("体重(lbs)",   lambda m: m.get("weight")),
        ("投球",        lambda m: m.get("pitch_hand")),
        ("打撃",        lambda m: m.get("bat_side")),
        ("MLBデビュー", lambda m: m.get("debut")),
        ("ニックネーム",lambda m: m.get("nickname")),
        ("比較年度",    lambda m: m.get("season")),
    ]
    for label, fn in fields:
        row = {"項目": label}
        for b in bundles:
            mlb = b.get("mlb") or {}
            row[b["player_name"]] = fn(mlb) or ""
        rows.append(row)
    return pd.DataFrame(rows, columns=["項目"] + names)

print("✅ 比較表関数定義完了")

✅ 比較表関数定義完了


In [9]:
# ─────────────────────────────────────
# 7. ローカルLLM分析 (Ollama + フォールバック)
# ─────────────────────────────────────
def call_ollama(prompt, model=OLLAMA_MODEL, timeout=120):
    try:
        payload = {"model": model, "prompt": prompt, "stream": False}
        r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        r.raise_for_status()
        return r.json().get("response", "")
    except Exception:
        return None

def rule_based_analysis(bundles):
    lines = ["## 📊 自動分析（ルールベース）\n"]
    names = [b["player_name"] for b in bundles]
    season = bundles[0].get("season") if bundles else None

    lines.append(f"**対象選手**: {', '.join(names)}\n")
    if season:
        lines.append(f"**比較年度**: {season}\n")

    ops_data = []
    for b in bundles:
        mlb = b.get("mlb") or {}
        ops_s = safe_num(mlb.get("season_hitting",{}).get("ops"))
        ops_c = safe_num(mlb.get("career_hitting",{}).get("ops"))
        ops_data.append((b["player_name"], ops_s, ops_c))

    lines.append("### 打撃指標比較")
    for name, ops_s, ops_c in ops_data:
        if ops_s:
            lines.append(f"- **{name}**: 今季OPS {ops_s:.3f}, 通算OPS {f'{ops_c:.3f}' if ops_c else 'N/A'}")

    best_ops = max(ops_data, key=lambda x: x[1] or 0)
    if best_ops[1]:
        lines.append(f"\n→ 今季OPS最高: **{best_ops[0]}** ({best_ops[1]:.3f})")

    avg_data = []
    for b in bundles:
        mlb = b.get("mlb") or {}
        avg = safe_num(mlb.get("season_hitting",{}).get("avg"))
        if avg:
            avg_data.append((b["player_name"], avg))
    if avg_data:
        best_avg = max(avg_data, key=lambda x: x[1])
        lines.append(f"→ 今季打率最高: **{best_avg[0]}** ({best_avg[1]:.3f})")

    hr_data = []
    for b in bundles:
        mlb = b.get("mlb") or {}
        hr = safe_num(mlb.get("season_hitting",{}).get("homeRuns"))
        if hr is not None:
            hr_data.append((b["player_name"], hr))
    if hr_data:
        best_hr = max(hr_data, key=lambda x: x[1])
        lines.append(f"→ 今季本塁打最多: **{best_hr[0]}** ({int(best_hr[1])}本)")

    era_data = []
    for b in bundles:
        mlb = b.get("mlb") or {}
        era = safe_num(mlb.get("season_pitching",{}).get("era"))
        if era is not None:
            era_data.append((b["player_name"], era))
    if era_data:
        lines.append("\n### 投球指標比較")
        best_era = min(era_data, key=lambda x: x[1])
        lines.append(f"→ 今季防御率最低（優秀）: **{best_era[0]}** ({best_era[1]:.2f})")

    lines.append("\n### 選手プロフィール要約")
    for b in bundles:
        wiki = b.get("wikipedia")
        if wiki:
            extract = wiki.get("extract", "")
            summary = extract[:300] + "..." if len(extract) > 300 else extract
            lines.append(f"\n**{b['player_name']}**: {summary}")

    return "\n".join(lines)

def llm_analysis(bundles):
    season = bundles[0].get("season") if bundles else None

    player_summaries = []
    for b in bundles:
        summary_parts = [f"選手: {b['player_name']}"]
        if season:
            summary_parts.append(f"比較年度: {season}")

        wiki = b.get("wikipedia")
        if wiki:
            summary_parts.append(f"Wikipedia概要: {truncate_text(wiki.get('extract',''), 400)}")
            summary_parts.append(f"Wikipediaタイトル: {wiki.get('title','')}")

        mlb = b.get("mlb")
        if mlb:
            sh = mlb.get("season_hitting",{})
            sp = mlb.get("season_pitching",{})
            sf = mlb.get("season_fielding",{})
            ch = mlb.get("career_hitting",{})
            cp = mlb.get("career_pitching",{})
            cf = mlb.get("career_fielding",{})

            if sh:
                summary_parts.append(
                    f"今季打撃: 打率{sh.get('avg','')}, 本塁打{sh.get('homeRuns','')}, "
                    f"打点{sh.get('rbi','')}, OPS{sh.get('ops','')}, OBP{sh.get('obp','')}, SLG{sh.get('slg','')}"
                )
            if ch:
                summary_parts.append(
                    f"通算打撃: 打率{ch.get('avg','')}, OPS{ch.get('ops','')}, 本塁打{ch.get('homeRuns','')}, 打点{ch.get('rbi','')}"
                )
            if sp:
                summary_parts.append(
                    f"今季投球: ERA{sp.get('era','')}, WHIP{sp.get('whip','')}, K{sp.get('strikeOuts','')}, W{sp.get('wins','')}"
                )
            if cp:
                summary_parts.append(
                    f"通算投球: ERA{cp.get('era','')}, WHIP{cp.get('whip','')}, K{cp.get('strikeOuts','')}, W{cp.get('wins','')}"
                )
            if sf:
                summary_parts.append(
                    f"今季守備: 守備率{sf.get('fielding','')}, 刺殺{sf.get('putOuts','')}, 補殺{sf.get('assists','')}, 失策{sf.get('errors','')}"
                )
            if cf:
                summary_parts.append(
                    f"通算守備: 守備率{cf.get('fielding','')}, 刺殺{cf.get('putOuts','')}, 補殺{cf.get('assists','')}, 失策{cf.get('errors','')}"
                )

        player_summaries.append("\n".join(summary_parts))

    prompt = f"""あなたはMLB専門の野球アナリストです。以下のMLB選手データをもとに、日本語で詳しく分析してください。

【比較年度】
{season if season else "指定なし"}

【選手データ】
{'\n---\n'.join(player_summaries)}

以下の観点で分析してください:
1. 各選手の強みと弱みの分析
2. 打撃・投球・守備の総合評価（野手/投手それぞれ適切に）
3. 選手間の比較と優劣
4. MLBでのキャリア的な評価
5. 総合的な結論

500文字以上で詳細に分析してください。"""

    print("🤖 ローカルLLM (Ollama) で分析中...")
    response = call_ollama(prompt)

    if response:
        return f"## 🤖 AI分析（ローカルLLM: {OLLAMA_MODEL}）\n\n{response}"
    else:
        print("⚠️ Ollama接続失敗 → ルールベース分析にフォールバック")
        return rule_based_analysis(bundles)

print("✅ LLM分析関数定義完了")

✅ LLM分析関数定義完了


In [10]:
# ─────────────────────────────────────
# 8. レポート生成
# ─────────────────────────────────────
def generate_full_report(bundles):
    names = [b["player_name"] for b in bundles]
    season = bundles[0].get("season") if bundles else None
    sections = []

    sections.append("# ⚾ MLB選手比較レポート")
    sections.append(f"**対象選手**: {' vs '.join(names)}")
    if season:
        sections.append(f"**比較年度**: {season}")
    sections.append(f"**生成日時**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    sections.append("")

    sections.append("## 📋 プロフィール比較")
    basic_df = build_basic_comparison(bundles)
    sections.append(basic_df.to_markdown(index=False) if not basic_df.empty else "データなし")

    hit_s = build_comparison_table(bundles, "season_hitting", MLB_HIT_FIELDS)
    if not hit_s.empty:
        sections.append("\n## ⚔️ 今季打撃成績（全指標）")
        sections.append(hit_s.to_markdown(index=False))

    hit_c = build_comparison_table(bundles, "career_hitting", MLB_HIT_FIELDS)
    if not hit_c.empty:
        sections.append("\n## 📈 通算打撃成績（全指標）")
        sections.append(hit_c.to_markdown(index=False))

    pit_s = build_comparison_table(bundles, "season_pitching", MLB_PITCH_FIELDS)
    if not pit_s.empty:
        sections.append("\n## 🎯 今季投球成績（全指標）")
        sections.append(pit_s.to_markdown(index=False))

    pit_c = build_comparison_table(bundles, "career_pitching", MLB_PITCH_FIELDS)
    if not pit_c.empty:
        sections.append("\n## 📊 通算投球成績（全指標）")
        sections.append(pit_c.to_markdown(index=False))

    fld_s = build_comparison_table(bundles, "season_fielding", MLB_FIELD_FIELDS)
    if not fld_s.empty:
        sections.append("\n## 🧤 今季守備成績")
        sections.append(fld_s.to_markdown(index=False))

    fld_c = build_comparison_table(bundles, "career_fielding", MLB_FIELD_FIELDS)
    if not fld_c.empty:
        sections.append("\n## 🧱 通算守備成績")
        sections.append(fld_c.to_markdown(index=False))

    sections.append("\n## 📖 Wikipedia概要")
    for b in bundles:
        wiki = b.get("wikipedia")
        if wiki:
            sections.append(f"\n### {b['player_name']}")
            sections.append(f"**出典**: [{wiki.get('title','')}]({wiki.get('fullurl','')})")
            sections.append(truncate_text(wiki.get("extract",""), 800))

    sections.append("\n---")
    analysis = llm_analysis(bundles)
    sections.append(analysis)

    return "\n\n".join(sections)

def save_report(text, player_names):
    output_dir = Path.cwd() / "baseball_reports"
    output_dir.mkdir(exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    name_slug = "_vs_".join([n.replace(" ","_") for n in player_names])
    filename = f"MLB_{name_slug}_{ts}.md"
    filepath = output_dir / filename
    filepath.write_text(text, encoding="utf-8")
    return filepath

print("✅ レポート生成関数定義完了")

✅ レポート生成関数定義完了


In [11]:
# ─────────────────────────────────────
# 9. UI 構築（メインロジック / MLB専用）
# ─────────────────────────────────────

state = {"bundles": [], "report": "", "league": "MLB", "season": DEFAULT_SEASON}

title_html = widgets.HTML(
    "<h2 style='color:#1e3a5f;border-bottom:2px solid #1e3a5f;padding-bottom:6px'>⚾ BaseballStatsMLB.com</h2>"
)

season_widget = widgets.BoundedIntText(
    value=DEFAULT_SEASON, min=1900, max=2100, step=1,
    description="比較年度:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

num_players_widget = widgets.BoundedIntText(
    value=2, min=2, max=9, step=1,
    description="比較選手数:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

player_inputs_box = widgets.VBox([])

run_btn = widgets.Button(
    description="🔍 データ取得＆分析",
    button_style="success",
    layout=widgets.Layout(width="220px", height="40px")
)

save_btn = widgets.Button(
    description="💾 結果を保存",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px")
)
save_btn.disabled = True

status_label = widgets.HTML("")

output_area = widgets.Output(layout=widgets.Layout(
    border="1px solid #ddd", padding="10px", min_height="200px"
))

def update_player_inputs(*args):
    n = num_players_widget.value
    state["league"] = "MLB"
    state["season"] = season_widget.value

    old_widgets = player_inputs_box.children
    old_values = [w.children[1].value for w in old_widgets if len(w.children) > 1]

    new_inputs = []
    for i in range(n):
        old_val = old_values[i] if i < len(old_values) else ""
        label = widgets.Label(f"選手 {i+1}:", layout=widgets.Layout(width="80px"))
        text = widgets.Text(
            value=old_val,
            placeholder="例: Shohei Ohtani",
            layout=widgets.Layout(width="400px")
        )
        new_inputs.append(widgets.HBox([label, text]))
    player_inputs_box.children = new_inputs

num_players_widget.observe(update_player_inputs, names="value")
season_widget.observe(update_player_inputs, names="value")
update_player_inputs()

def on_run_clicked(b):
    player_names = []
    for row in player_inputs_box.children:
        name = row.children[1].value.strip()
        if name:
            player_names.append(name)

    if len(player_names) < 2:
        status_label.value = "<span style='color:red'>⚠️ 2人以上の選手名を入力してください</span>"
        return

    season = season_widget.value
    state["league"] = "MLB"
    state["season"] = season

    run_btn.disabled = True
    save_btn.disabled = True
    status_label.value = "<span style='color:#1e3a5f'>⏳ MLBデータ取得中...</span>"

    with output_area:
        clear_output()
        display(Markdown(f"## ⏳ MLB データ取得中...\n比較年度: {season}\n選手: {', '.join(player_names)}"))

    try:
        bundles = []
        for i, name in enumerate(player_names):
            status_label.value = f"<span style='color:#1e3a5f'>⏳ [{i+1}/{len(player_names)}] {name} 取得中...</span>"
            bundle = build_player_bundle(name, season=season)
            bundles.append(bundle)
            time.sleep(0.5)

        state["bundles"] = bundles

        status_label.value = "<span style='color:#1e3a5f'>🤖 レポート生成中（LLM分析含む）...</span>"
        report = generate_full_report(bundles)
        state["report"] = report

        with output_area:
            clear_output()
            display(Markdown(report))

        save_btn.disabled = False
        status_label.value = "<span style='color:green'>✅ 分析完了！「結果を保存」ボタンで保存できます</span>"

    except Exception as e:
        with output_area:
            clear_output()
            display(Markdown(f"**❌ エラーが発生しました**: `{e}`\n\n詳細: 選手名のスペルを確認してください。"))
        status_label.value = f"<span style='color:red'>❌ エラー: {e}</span>"
    finally:
        run_btn.disabled = False

def on_save_clicked(b):
    if not state["report"]:
        status_label.value = "<span style='color:red'>先に分析を実行してください</span>"
        return
    try:
        names = [b["player_name"] for b in state["bundles"]]
        filepath = save_report(state["report"], names)
        status_label.value = f"<span style='color:green'>💾 保存完了: {filepath}</span>"
        with output_area:
            display(Markdown(f"\n\n---\n\n💾 **保存完了**: `{filepath}`"))
    except Exception as e:
        status_label.value = f"<span style='color:red'>❌ 保存エラー: {e}</span>"

run_btn.on_click(on_run_clicked)
save_btn.on_click(on_save_clicked)

settings_box = widgets.VBox([
    widgets.HTML("<b>① 比較年度を入力</b>"),
    season_widget,
    widgets.HTML("<b>② 比較する選手数を入力</b>"),
    num_players_widget,
    widgets.HTML("<b>③ MLB選手名を入力</b>"),
    player_inputs_box,
    widgets.HTML("<b>④ 実行・保存</b>"),
    widgets.HBox([run_btn, save_btn]),
    status_label,
], layout=widgets.Layout(
    border="1px solid #ccc",
    padding="15px",
    margin="10px 0",
    border_radius="8px"
))

ui = widgets.VBox([
    title_html,
    settings_box,
    widgets.HTML("<h3>📊 分析結果</h3>"),
    output_area
])

print("✅ UI構築完了")
print("\n🚀 以下のUIを操作してください:")
display(ui)

✅ UI構築完了

🚀 以下のUIを操作してください:
